In [2]:
import pickle
import os
from pathlib import Path
from collections import defaultdict

import numpy as np
from loguru import logger
import matplotlib.pyplot as plt

from behave_analysis.process.session import get_experiment
from behave_analysis.analyze.overview.homing_analysis.decoding_spatial_eff.helper.model_escapes import run_escape_analysis, analyze_behavioral_contribution
# from behave_analysis.analyze.overview.homing_analysis.decoding_spatial_eff.helper.data_gen_escapes import produce_data, create_the_past_design_matrix

# Importing the mice sessions to use

In [3]:
from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept
from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept
from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept
from behave_analysis.database.Experiments.JAL006_ex import JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, JAL6_flip7_1apr
from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr,JAL7_30apr
from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_tiny_3may, JAL8_flip4_10may, JAL8_14may, JAL8_21may, JAL8_flip3_7may

# Defined session names, experiments and hash mice to sessions

In [4]:
# The values must be in order of data for the code to work
mice_groups = {
    "JAL6": ['JAL6_flip3_18mar', 'JAL6_flip4_21mar', 'JAL6_flip5_25mar', 'JAL6_28mar'],
    "JAL3": ['JAL3_25aug', 'JAL3_1sept', 'JAL3_4sept', 'JAL3_7sept'],
    "JAL7": ['JAL7_flip2_12mar', 'JAL7_flip5_22mar', 'JAL7_sesh8_9apr', 'JAL7_sesh9_16apr', 'JAL7_23apr'],
    "JAL8": ['JAL8_flip1_25apr', 'JAL8_flip2_29apr', 'JAL8_flip4_10may', "JAL8_flip3_7may", 'JAL8_14may'],
    "JAL4": ['JAL4_28aug', 'JAL4_3rdSept', 'JAL4_11thSept', 'JAL4_19thSept'],
    "JAL5": ['JAL5_8thSept', 'JAL5_21stSept']}

# The experiment objects index must match the session name index for the code to work
experiments_objects = [JAL6_flip3_18mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_28mar,
                       JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept,
                       JAL005_8thSept, JAL005_21stSept,
                       JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr,
                       JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may, JAL8_flip3_7may,
                       JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept]

# The session name indexes must match the experiment object indexes
session_names = ["JAL6_flip3_18mar", "JAL6_flip4_21mar", "JAL6_flip5_25mar", "JAL6_28mar",
                 "JAL3_25aug", "JAL3_1sept", "JAL3_4sept", "JAL3_7sept",
                 "JAL5_8thSept", "JAL5_21stSept",
                 "JAL7_sesh8_9apr", "JAL7_sesh9_16apr", "JAL7_flip5_22mar", "JAL7_flip2_12mar", "JAL7_23apr",
                 "JAL8_flip1_25apr", "JAL8_flip2_29apr", "JAL8_flip4_10may", "JAL8_14may", "JAL8_flip3_7may",
                 "JAL4_3rdSept", "JAL4_19thSept", "JAL4_28aug", "JAL4_11thSept"]

If you haven't created the data already, the produce data function will create the relevant design matrices and classes for logistic regression to run. The last arugmnet, which is a function specific to how the design matrix is created, determines whether we look in the past before the mouse has done the homings, or whether we want to run the decoding during the homing. Only the option looking before the homing has started is included, as this minimises spurious correlations with behaviour. 

In [ ]:
#produce_data(experiments_objects, session_names, "compute_the_first_20_frames_before_homing", create_the_past_design_matrix)


Load the data that has previously been generated

In [5]:
path = "Z:\Jasmine_Laurence\single_trial_overview\decoding_spatial_efficiency\escapes"
dir = Path(path)
name_of_data = "escapes_first_20_frames"
path = dir / f"{name_of_data}.pkl"
with open(path, "rb") as f:
    compute_the_first_20_frames_before_escape =  pickle.load(f)

Create the accuracy data if it does not exsist, this is done if you want to run a control for example, by shifting the neural data in the produce data step, we would want to recalculate the accuracy data

In [7]:
print(compute_the_first_20_frames_before_escape.keys())

dict_keys(['JAL6_flip3_18mar', 'JAL6_flip4_21mar', 'JAL6_flip5_25mar', 'JAL6_28mar', 'JAL3_25aug', 'JAL3_1sept', 'JAL3_4sept', 'JAL3_7sept', 'JAL5_8thSept', 'JAL5_21stSept', 'JAL7_sesh8_9apr', 'JAL7_sesh9_16apr', 'JAL7_flip5_22mar', 'JAL7_flip2_12mar', 'JAL7_23apr', 'JAL8_flip1_25apr', 'JAL8_flip2_29apr', 'JAL8_flip4_10may', 'JAL8_14may', 'JAL8_flip3_7may', 'JAL4_3rdSept', 'JAL4_19thSept', 'JAL4_28aug', 'JAL4_11thSept'])


In [10]:

import os
import pickle
from collections import defaultdict
from pathlib import Path

from loguru import logger
import pandas as pd
import scipy.stats as stats
import numpy as np
from sklearn.utils import resample
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

from behave_analysis.utils.creating_directories import make_directory
from behave_analysis.analyze.overview.homing_analysis.decoding_spatial_eff.helper.data_gen import load_video_data

def compute_accuracy_data_escapes_with_mapping(data_type, data_type_name, experiments_objects, session_names, random_labels=False):
    """Compute and analyze decoding accuracy data using logistic regression for a set of escape sessions.
    
    This version uses a direct mapping between session names and experiment objects.
    
    Args:
        data_type (dict): A dictionary where keys are session names and values are session-specific data
        data_type_name (str): Name of the data type for saving results
        experiments_objects (list): List of experiment objects
        session_names (list): List of session names in the same order as experiments_objects
        random_labels (bool): Whether to use random labels for control comparison
        
    Returns:
        dict: Dictionary containing accuracy data, KS stats, and model coefficients
    """
    save_base = r"Z:\Jasmine_Laurence\single_trial_overview"
    save_path = os.path.join(save_base, "decoding_spatial_efficiency", data_type_name)
    dir = make_directory(save_path)
    whole_escape = {}
    ks_decoding_comparison = {}
    coefs = {}

    # Create mapping from session names to experiment objects
    session_to_exp = {name: obj for name, obj in zip(session_names, experiments_objects)}
    
    # Log the mapping information
    logger.info(f"Created mapping for {len(session_to_exp)} sessions to experiment objects")
    
    # Process each session in the data
    for session_name, data in data_type.items():
        # Find the corresponding experiment object
        if session_name not in session_names:
            logger.warning(f"Session {session_name} not in the provided session names list, skipping")
            continue
            
        # Get the experiment object for this session
        experiment = session_to_exp[session_name]
        logger.info(f"Running logistic regression for session: {session_name}")
        
        try:
            video_df = load_video_data(experiment)
            if video_df is None:
                logger.warning(f"Failed to load video data for session {session_name}, skipping")
                continue
                
            design_matrix = data["design_matrix"]  # The X data
            classes_extended = np.asarray(data["classes_extended"])  # The y data
            
            # Check if escape_ids is present in the data
            group_data = data.get("escape_ids", data.get("homing_ids", None))
            if group_data is None:
                logger.warning(f"No group data (escape_ids) found for session {session_name}, skipping")
                continue
                
            if random_labels:
                np.random.shuffle(classes_extended)  # randomly shuffle the classes
                
            xBalanced, y_balanced, groups_balanced = handle_class_imbalance(
                classes_extended, session_name, design_matrix, {"escape_ids": group_data}
            )
            
            if xBalanced is None:  # If the cutoff for the amount of data has not been met
                continue  # Skip this session

            # Initialize session data dictionaries if not already present
            if session_name not in ks_decoding_comparison:
                ks_decoding_comparison[session_name] = {}
            if session_name not in coefs:
                coefs[session_name] = {}
                
            group_kfold = GroupKFold(n_splits=2)  # k-fold iterator variation with non-overlapping groups
            fold_accuracies = []

            # Assign and remove the last column of the design matrix - Frame numbers are needed to access the behavior data
            frame_numbers = xBalanced[:, -1].astype(int)  # Access the last column of the design matrix
            xBalanced = xBalanced[:, :-1]  # Remove the last column of the design matrix

            for fold, (train_index, test_index) in enumerate(group_kfold.split(xBalanced, y_balanced, groups_balanced)):
                X_train, X_test = xBalanced[train_index], xBalanced[test_index]
                y_train, y_test = y_balanced[train_index], y_balanced[test_index]
                
                if not check_there_are_two_classes(y_train):
                    break
                    
                down_count_ans = count_class_labels(y_train)
                logger.info(f"Class distribution for fold: {fold}. 0: {down_count_ans.get(0, 0)}, 1: {down_count_ans.get(1, 0)}")
                
                model = LogisticRegression(
                    class_weight="balanced",
                    random_state=1337,
                    max_iter=10000,
                ).fit(X_train, y_train)
                
                y_pred = model.predict(X_test)
                accuracy = accuracy_score(y_test, y_pred)
                fold_accuracies.append(accuracy)
                
                # Plot behavioral data distributions for training set
                try:
                    x_ks, y_ks, s_ks, h_ks = plot_class_distributions_and_compute_kolmogorov_Smirnov(
                        frame_numbers, video_df, y_train, dir, session_name, fold, accuracy
                    )
                    ks_decoding_comparison[session_name][fold] = {
                        "x": x_ks, "y": y_ks, "speed": s_ks, "hdir": h_ks, "accuracy": accuracy
                    }
                except Exception as e:
                    logger.error(f"Error plotting class distributions for session {session_name}, fold {fold}: {e}")
                    ks_decoding_comparison[session_name][fold] = {"accuracy": accuracy}
                    
                # Store model coefficients
                coefs[session_name][fold] = model.coef_[0]

            if fold_accuracies:
                whole_escape[session_name] = np.mean(fold_accuracies)
                logger.info(f"Mean accuracy for session {session_name}: {whole_escape[session_name]}")
            
        except Exception as e:
            logger.error(f"Error processing session {session_name}: {e}")
            continue

    # Save all results
    save_data = {
        "accuracy_data": whole_escape, 
        "ks_data": ks_decoding_comparison, 
        "coefs": coefs
    }
    
    with open(dir / Path(f"{data_type_name}_accuracy_ks_coeffs_data.pkl"), "wb") as f:
        pickle.dump(save_data, f)

    return save_data


def compute_down_sampled_accuracy_data_escapes_with_mapping(data_type, experiments_objects, session_names, behaviour_to_subsample: str):
    """Compute accuracy data with behavioral subsampling for escape analysis, using explicit mapping.
    
    Args:
        data_type (dict): Dictionary of session data
        experiments_objects (list): List of experiment objects
        session_names (list): List of session names in the same order as experiments_objects
        behaviour_to_subsample (str): Behavioral variable to subsample by
        
    Returns:
        dict: Dictionary of mean accuracies by session
    """
    accuracy_data = {}

    # Create mapping from session names to experiment objects
    session_to_exp = {name: obj for name, obj in zip(session_names, experiments_objects)}
    
    # Process each session in the data
    for session_name, data in data_type.items():
        # Check if session is in our mapping
        if session_name not in session_names:
            logger.warning(f"Session {session_name} not in the provided session names list, skipping")
            continue
        
        # Get the experiment object
        experiment = session_to_exp[session_name]
        logger.info(f"Running subsampled logistic regression for session: {session_name}")
        
        try:
            design_matrix = data["design_matrix"]  # The X data
            classes_extended = np.asarray(data["classes_extended"])  # The y data
            
            # Check if escape_ids is present in the data
            group_data = data.get("escape_ids", data.get("homing_ids", None))
            if group_data is None:
                logger.warning(f"No group data (escape_ids) found for session {session_name}, skipping")
                continue
                
            xBalanced, y_balanced, groups_balanced = handle_class_imbalance(
                classes_extended, session_name, design_matrix, {"escape_ids": group_data}, cutoff=400  # 10s of data
            )
            
            video_df = load_video_data(experiment)
            if video_df is None or xBalanced is None:
                continue  # Skip this session
                
            group_kfold = GroupKFold(n_splits=2)
            fold_accuracies = []
            frame_numbers = xBalanced[:, -1].astype(int)
            xBalanced = xBalanced[:, :-1]  # Remove the last column of the design matrix

            for fold, (train_index, test_index) in enumerate(group_kfold.split(xBalanced, y_balanced, groups_balanced)):
                try:
                    # Find frames with good and bad labels in training set
                    good_indices = np.where(y_balanced[train_index] == 1)[0]
                    bad_indices = np.where(y_balanced[train_index] == 0)[0]
                    good_frames = frame_numbers[train_index][good_indices]
                    bad_frames = frame_numbers[train_index][bad_indices]
                    
                    # Get behavioral values for the selected variable
                    if behaviour_to_subsample not in video_df.columns:
                        logger.warning(f"Behavior variable {behaviour_to_subsample} not found in video_df for {session_name}")
                        continue
                        
                    good_behavior_values = video_df[behaviour_to_subsample][good_frames].values
                    bad_behavior_values = video_df[behaviour_to_subsample][bad_frames].values
                    
                    # Create bins and find overlap between distributions
                    min_val = min(np.min(good_behavior_values), np.min(bad_behavior_values))
                    max_val = max(np.max(good_behavior_values), np.max(bad_behavior_values))
                    bins = np.linspace(min_val, max_val, 25)
                    
                    good_hist, _ = np.histogram(good_behavior_values, bins=bins)
                    bad_hist, _ = np.histogram(bad_behavior_values, bins=bins)
                    overlap = np.minimum(good_hist, bad_hist)

                    # Find indices of frames in overlapping regions of the distributions
                    overlap_frames = []
                    for i in range(len(bins) - 1):
                        if overlap[i] > 0:
                            # Identify frames in this bin for both classes
                            bin_mask_good = (good_behavior_values >= bins[i]) & (good_behavior_values < bins[i + 1])
                            bin_mask_bad = (bad_behavior_values >= bins[i]) & (bad_behavior_values < bins[i + 1])
                            
                            # Get frames in this bin
                            good_bin_frames = good_frames[bin_mask_good]
                            bad_bin_frames = bad_frames[bin_mask_bad]
                            
                            # Add to our collection
                            overlap_frames.extend(good_bin_frames.tolist())
                            overlap_frames.extend(bad_bin_frames.tolist())

                    # Get indices in the original data matrix for the overlapping frames
                    overlap_indices = np.where(np.isin(frame_numbers[train_index], overlap_frames))[0]
                    
                    # Skip if no overlap was found
                    if len(overlap_indices) == 0:
                        logger.warning(f"No behavioral overlap found for {session_name}, fold {fold}")
                        continue
                        
                    # Extract balanced data
                    X_train_balanced = xBalanced[train_index][overlap_indices]
                    y_train_balanced = y_balanced[train_index][overlap_indices]
                    X_test = xBalanced[test_index]
                    y_test = y_balanced[test_index]

                    if not check_there_are_two_classes(y_train_balanced):
                        logger.warning(f"Not enough class diversity in fold {fold} for session {session_name}")
                        continue

                    # Check class balance after subsampling
                    down_count_ans = count_class_labels(y_train_balanced)
                    logger.info(f"Subsampled class distribution - fold {fold}: 0: {down_count_ans.get(0, 0)}, 1: {down_count_ans.get(1, 0)}")
                    
                    # Train model and compute accuracy
                    model = LogisticRegression(
                        class_weight="balanced",
                        random_state=1337,
                        max_iter=10000,
                    ).fit(X_train_balanced, y_train_balanced)
                    
                    y_pred = model.predict(X_test)
                    accuracy = accuracy_score(y_test, y_pred)
                    fold_accuracies.append(accuracy)
                    
                except Exception as e:
                    logger.error(f"Error in fold {fold} for session {session_name}: {e}")
                    continue

            if fold_accuracies:
                accuracy_data[session_name] = np.mean(fold_accuracies)
                logger.info(f"Mean accuracy for session {session_name}: {accuracy_data[session_name]}")
            else:
                logger.warning(f"No valid folds for session {session_name}")
                
        except Exception as e:
            logger.error(f"Error processing session {session_name}: {e}")
            continue

    return accuracy_data


def run_escape_analysis_with_mapping(escape_data_path, experiments_objects, session_names, mice_groups, random_comparison=True):
    """Run the full escape analysis pipeline using explicit session-experiment mapping.
    
    Args:
        escape_data_path (str): Path to the pickle file containing escape data
        experiments_objects (list): List of experiment objects
        session_names (list): List of session names matching experiment_objects
        mice_groups (dict): Dictionary mapping mouse IDs to ordered lists of session names
        random_comparison (bool): Whether to run a random label comparison
        
    Returns:
        dict: Analysis results
    """
    # Load the escape data
    try:
        with open(escape_data_path, "rb") as f:
            escape_data = pickle.load(f)
    except Exception as e:
        logger.error(f"Error loading escape data: {e}")
        return None
        
    # Get the data type name from the file path
    data_type_name = Path(escape_data_path).stem
    
    # Print basic info about the data
    logger.info(f"Loaded {len(escape_data)} sessions from {escape_data_path}")
    if escape_data:
        first_session = next(iter(escape_data))
        logger.info(f"Sample session {first_session} has keys: {escape_data[first_session].keys()}")
    
    # Run the main accuracy computation with mapping
    logger.info(f"Computing accuracy for {data_type_name}...")
    results = compute_accuracy_data_escapes_with_mapping(escape_data, data_type_name, experiments_objects, session_names)
    
    # Run random label comparison if requested
    if random_comparison:
        logger.info(f"Computing accuracy with random labels for {data_type_name}...")
        random_results = compute_accuracy_data_escapes_with_mapping(
            escape_data, f"{data_type_name}_random", experiments_objects, session_names, random_labels=True
        )
        
        # Compare real vs. random accuracy
        real_accs = list(results['accuracy_data'].values())
        random_accs = list(random_results['accuracy_data'].values())
        
        if real_accs and random_accs:
            logger.info(f"Average real accuracy: {np.mean(real_accs):.3f} ± {np.std(real_accs):.3f}")
            logger.info(f"Average random accuracy: {np.mean(random_accs):.3f} ± {np.std(random_accs):.3f}")
            
            # Statistical test if we have enough data
            if len(real_accs) >= 5 and len(random_accs) >= 5:
                t_stat, p_val = stats.ttest_ind(real_accs, random_accs)
                logger.info(f"T-test: t={t_stat:.3f}, p={p_val:.3f}")
        
        # Plot real vs. random comparison
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.boxplot([real_accs, random_accs], labels=["Real Labels", "Random Labels"])
        ax.set_ylabel("Accuracy")
        ax.set_title(f"Escape Decoding Accuracy: Real vs. Random ({data_type_name})")
        ax.axhline(y=0.5, color='r', linestyle=':', label="Chance")
        fig.savefig(f"escape_accuracy_comparison_{data_type_name}.png")
        plt.close(fig)
    
    # Plot accuracy across sessions
    if results['accuracy_data']:
        try:
            fig = plot_accuracy_across_sessions(
                results['accuracy_data'], 
                mice_groups,
                plot_title=f"Escape Decoding - {data_type_name}"
            )
            fig.savefig(f"escape_accuracy_across_sessions_{data_type_name}.png")
            plt.close(fig)
        except Exception as e:
            logger.error(f"Error plotting accuracy across sessions: {e}")
    
    return results


def analyze_behavioral_contribution_with_mapping(escape_data_path, experiments_objects, session_names, behaviors=None):
    """Analyze behavioral contributions with explicit session-experiment mapping.
    
    Args:
        escape_data_path (str): Path to the escape data file
        experiments_objects (list): List of experiment objects
        session_names (list): List of session names matching experiment_objects
        behaviors (list, optional): List of behavioral variables to analyze. 
                                   Defaults to ["mouse_x_position", "mouse_y_position", "speed", "hdir"]
    
    Returns:
        dict: Analysis results by behavior
    """
    # Set default behaviors if none provided
    if behaviors is None:
        behaviors = ["mouse_x_position", "mouse_y_position", "speed", "hdir"]
    
    # Load the escape data
    try:
        with open(escape_data_path, "rb") as f:
            escape_data = pickle.load(f)
    except Exception as e:
        logger.error(f"Error loading escape data: {e}")
        return None
    
    # Get normal accuracy as baseline
    data_type_name = Path(escape_data_path).stem
    baseline_results = compute_accuracy_data_escapes_with_mapping(
        escape_data, f"{data_type_name}_baseline", experiments_objects, session_names
    )
    
    # Store all results
    all_results = {
        "baseline": baseline_results["accuracy_data"]
    }
    
    # Analyze each behavior variable
    for behavior in behaviors:
        logger.info(f"Analyzing contribution of {behavior}...")
        behavior_results = compute_down_sampled_accuracy_data_escapes_with_mapping(
            escape_data, experiments_objects, session_names, behavior
        )
        all_results[behavior] = behavior_results
        
        # Compare with baseline if we have results
        if behavior_results and baseline_results["accuracy_data"]:
            # Get sessions that appear in both results
            common_sessions = set(behavior_results.keys()) & set(baseline_results["accuracy_data"].keys())
            
            if common_sessions:
                baseline_accs = [baseline_results["accuracy_data"][s] for s in common_sessions]
                behavior_accs = [behavior_results[s] for s in common_sessions]
                
                # Calculate differences
                mean_baseline = np.mean(baseline_accs)
                mean_behavior = np.mean(behavior_accs)
                
                logger.info(f"Baseline accuracy: {mean_baseline:.3f}")
                logger.info(f"{behavior} controlled accuracy: {mean_behavior:.3f}")
                logger.info(f"Difference: {mean_baseline - mean_behavior:.3f}")
                
                # Statistical test if enough data
                if len(common_sessions) >= 5:
                    t_stat, p_val = stats.ttest_rel(baseline_accs, behavior_accs)
                    logger.info(f"Paired t-test: t={t_stat:.3f}, p={p_val:.3f}")
    
    # Plot comparison of all behaviors
    plot_behavioral_contributions(all_results, data_type_name)
    
    return all_results


# Example usage
def analyze_escape_data():
    """Run the complete escape data analysis pipeline."""
    # Define paths
    base_path = r"Z:\Jasmine_Laurence\single_trial_overview\decoding_spatial_efficiency\escapes"
    escape_data_files = [
        "escapes_first_20_frames.pkl",
        "escapes_entire_spatial_efficiency.pkl", 
        "escapes_random_times_200_frames.pkl"
    ]
    
    # Load experiment objects and session names from the global scope
    # These should be defined in your main script
    
    # Run the analysis for each data file
    results = {}
    for data_file in escape_data_files:
        data_path = os.path.join(base_path, data_file)
        logger.info(f"Analyzing escape data file: {data_file}")
        
        # Run the main accuracy analysis
        result = run_escape_analysis_with_mapping(
            data_path, 
            experiments_objects,  # This should be defined in your main script
            session_names,       # This should be defined in your main script
            mice_groups         # This should be defined in your main script
        )
        
        # Store the results
        data_name = Path(data_file).stem
        results[data_name] = result
        
        # Run behavioral contribution analysis
        behavioral_result = analyze_behavioral_contribution_with_mapping(
            data_path,
            experiments_objects,
            session_names
        )
        
        results[f"{data_name}_behavioral"] = behavioral_result
    
    return results

In [12]:
def count_class_labels(classes):
    """Count the number of instances for each class label."""
    dic_to_store = {}
    unique_classes = np.unique(classes)
    for cls in unique_classes:
        dic_to_store[cls] = np.sum(classes == cls)
    return dic_to_store


def downsample_larger_class(design_matrix, classes_extended, random_state, escape_ids):
    """Randomly downsample the larger class to match the smaller class to ensure balanced training."""
    X = design_matrix
    y = classes_extended
    class_0_indices = np.where(y == 0)[0]
    class_1_indices = np.where(y == 1)[0]

    # Ensure only downsampling of the larger class
    if len(class_0_indices) > len(class_1_indices):
        class_0_indices = resample(class_0_indices, replace=False, n_samples=len(class_1_indices), random_state=random_state)
    elif len(class_1_indices) > len(class_0_indices):
        class_1_indices = resample(class_1_indices, replace=False, n_samples=len(class_0_indices), random_state=random_state)

    # Combine indices from both classes
    balanced_indices = np.concatenate([class_0_indices, class_1_indices])

    # Extract balanced data
    xBalanced = X[balanced_indices]
    y_balanced = y[balanced_indices]

    # Handle group ids for cross-validation
    if isinstance(escape_ids, dict):
        groups_balanced = [escape_ids[i] for i in balanced_indices if i in escape_ids]
    else:
        # Assume escape_ids is a list or array that can be indexed directly
        groups_balanced = [escape_ids[i] for i in balanced_indices]

    return xBalanced, y_balanced, groups_balanced


def check_there_is_enough_data(count_ans, session_name, cutoff):
    """Returns True if there is enough data for both classes."""
    for cls in [0, 1]:
        if cls not in count_ans or count_ans[cls] < cutoff:
            logger.warning(f"For session {session_name}, class {cls} has less than {cutoff / 40} seconds of data, skipping session")
            return False
    return True


def check_there_are_two_classes(y_train):
    """Check if the training data contains examples from both classes."""
    if len(np.unique(y_train)) < 2:
        logger.warning("Skipping fold because there is only one class in the training data")
        return False
    return True


def handle_class_imbalance(classes_extended, session_name, design_matrix, data, cutoff=80):
    """Balance classes in the dataset by downsampling the larger class."""
    # Get group identifiers for cross-validation (escape_ids)
    group_data = data.get("escape_ids", data.get("homing_ids", None))
    if group_data is None:
        logger.warning(f"No group data found for session {session_name}")
        return None, None, None

    # Check the class distribution before down sampling
    count_ans = count_class_labels(classes_extended)
    logger.info(f"Class distribution before balancing - 0: {count_ans.get(0, 0)}, 1: {count_ans.get(1, 0)}")

    if not check_there_is_enough_data(count_ans, session_name, cutoff=cutoff):
        return None, None, None  # The cutoff has not been met

    # Down sample the larger class
    xBalanced, y_balanced, groups_balanced = downsample_larger_class(
        design_matrix=design_matrix, classes_extended=classes_extended, random_state=1337, escape_ids=group_data
    )

    count_of_downsampled_classes = count_class_labels(y_balanced)
    logger.info(f"Class distribution after balancing - 0: {count_of_downsampled_classes.get(0, 0)}, " 
               f"1: {count_of_downsampled_classes.get(1, 0)}")

    return xBalanced, y_balanced, groups_balanced


def are_the_distributions_different(good_data, bad_data):
    """Conduct a Kolmogorov-Smirnov test to see if the distributions are different."""
    # Convert inputs to numpy arrays for safety
    good_data = np.array(good_data)
    bad_data = np.array(bad_data)

    # Handle empty data
    if len(good_data) == 0 or len(bad_data) == 0:
        return False, 0.0, 1.0

    # Remove NaN values
    good_data = good_data[~np.isnan(good_data)]
    bad_data = bad_data[~np.isnan(bad_data)]

    # Ensure we have enough data after filtering NaNs
    if len(good_data) < 2 or len(bad_data) < 2:
        return False, 0.0, 1.0

    # Perform KS test
    try:
        ks_statistic, p_value = stats.ks_2samp(good_data, bad_data)
        return p_value < 0.05, ks_statistic, p_value
    except Exception as e:
        logger.error(f"Error in KS test: {e}")
        return False, 0.0, 1.0

def plot_class_distributions_and_compute_kolmogorov_Smirnov(frame_numbers, video_df, y_train, dir, session_name, fold, accuracy):
    """Plot behavioral distributions by class and compute KS statistics to check for differences."""
    # Create directory if it doesn't exist
    os.makedirs(dir, exist_ok=True)

    # Get the behavioral distributions of the classes for the training dataset
    good_indices = np.where(y_train == 1)[0]
    bad_indices = np.where(y_train == 0)[0]

    # Exit if either class is empty
    if len(good_indices) == 0 or len(bad_indices) == 0:
        logger.warning(f"Empty class in data for session {session_name}, fold {fold}")
        return 0.0, 0.0, 0.0, 0.0

    # Get frame numbers for both classes
    good_frames = frame_numbers[good_indices]
    bad_frames = frame_numbers[bad_indices]

    # Initialize arrays for behavioral variables
    good_x, bad_x = [], []
    good_y, bad_y = [], []
    good_speed, bad_speed = [], []
    good_hdir, bad_hdir = [], []

    # Extract behavioral data with error handling
    try:
        if "mouse_x_position" in video_df.columns:
            good_x = video_df["mouse_x_position"][good_frames].values
            bad_x = video_df["mouse_x_position"][bad_frames].values

        if "mouse_y_position" in video_df.columns:
            good_y = video_df["mouse_y_position"][good_frames].values
            bad_y = video_df["mouse_y_position"][bad_frames].values

        if "speed" in video_df.columns:
            good_speed = video_df["speed"][good_frames].values
            bad_speed = video_df["speed"][bad_frames].values

        if "hdir" in video_df.columns:
            good_hdir = video_df["hdir"][good_frames].values
            bad_hdir = video_df["hdir"][bad_frames].values
    except Exception as e:
        logger.error(f"Error extracting behavioral data: {e}")
        return 0.0, 0.0, 0.0, 0.0

    # Compute the Kolmogorov-Smirnov statistic
    x_stat, x_ks, _ = are_the_distributions_different(good_x, bad_x)
    y_stat, y_ks, _ = are_the_distributions_different(good_y, bad_y)
    speed_stat, s_ks, _ = are_the_distributions_different(good_speed, bad_speed)
    hdir_stat, h_ks, _ = are_the_distributions_different(good_hdir, bad_hdir)

    # Plot the distributions
    try:
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        axes = axes.flatten()

        # X position plot
        if len(good_x) > 0 and len(bad_x) > 0:
            axes[0].hist(good_x, bins=30, alpha=0.5, label="Good")
            axes[0].hist(bad_x, bins=30, alpha=0.5, label="Bad")
            axes[0].set_title(f"X Position Distribution - Diff: {x_stat}")
            axes[0].set_ylabel("Frequency")
            axes[0].legend()

        # Y position plot
        if len(good_y) > 0 and len(bad_y) > 0:
            axes[1].hist(good_y, bins=30, alpha=0.5, label="Good")
            axes[1].hist(bad_y, bins=30, alpha=0.5, label="Bad")
            axes[1].set_title(f"Y Position Distribution - Diff: {y_stat}")
            axes[1].set_ylabel("Frequency")
            axes[1].legend()

        # Speed plot
        if len(good_speed) > 0 and len(bad_speed) > 0:
            axes[2].hist(good_speed, bins=30, alpha=0.5, label="Good")
            axes[2].hist(bad_speed, bins=30, alpha=0.5, label="Bad")
            axes[2].set_title(f"Speed Distribution - Diff: {speed_stat}")
            axes[2].set_ylabel("Frequency")
            axes[2].legend()

        # Head direction plot
        if len(good_hdir) > 0 and len(bad_hdir) > 0:
            axes[3].hist(good_hdir, bins=30, alpha=0.5, label="Good")
            axes[3].hist(bad_hdir, bins=30, alpha=0.5, label="Bad")
            axes[3].set_title(f"Hdir Distribution - Diff: {hdir_stat}")
            axes[3].set_ylabel("Frequency")
            axes[3].legend()

        plt.suptitle(f"Behavioral Class Distributions - {session_name}, Fold {fold}, Acc: {accuracy:.2f}")
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.savefig(os.path.join(dir, f"class_distributions_behaviour_{session_name}_fold_{fold}.png"))
        plt.close()
    except Exception as e:
        logger.error(f"Error plotting distributions: {e}")

    return x_ks, y_ks, s_ks, h_ks

In [13]:
# Path to your generated escape data
escape_data_path = "Z:/Jasmine_Laurence/single_trial_overview/decoding_spatial_efficiency/escapes/escapes_first_20_frames.pkl"

results = run_escape_analysis_with_mapping(
    escape_data_path, 
    experiments_objects,
    session_names,
    mice_groups
)

2025-02-28 17:25:23.824 | INFO     | __main__:run_escape_analysis_with_mapping:319 - Loaded 24 sessions from Z:/Jasmine_Laurence/single_trial_overview/decoding_spatial_efficiency/escapes/escapes_first_20_frames.pkl
2025-02-28 17:25:23.825 | INFO     | __main__:run_escape_analysis_with_mapping:322 - Sample session JAL6_flip3_18mar has keys: dict_keys(['design_matrix', 'classes_extended', 'escape_ids'])
2025-02-28 17:25:23.825 | INFO     | __main__:run_escape_analysis_with_mapping:325 - Computing accuracy for escapes_first_20_frames...
2025-02-28 17:25:23.829 | INFO     | __main__:compute_accuracy_data_escapes_with_mapping:45 - Created mapping for 24 sessions to experiment objects
2025-02-28 17:25:23.829 | INFO     | __main__:compute_accuracy_data_escapes_with_mapping:56 - Running logistic regression for session: JAL6_flip3_18mar
2025-02-28 17:25:29.545 | INFO     | __main__:handle_class_imbalance:67 - Class distribution before balancing - 0: 40, 1: 140
2025-02-28 17:25:29.546 | WARNING 

# Load prior accuracy and coefficients

In [7]:
with open(r"Z:\Jasmine_Laurence\single_trial_overview\decoding_spatial_efficiency\500ms_before_accuracy\500ms_before_accuracy_accuracy_ks_coeffs_data.pkl", "rb") as f:
    accuracy_ks_coeffs_data = pickle.load(f)
model_coeffs = accuracy_ks_coeffs_data["coefs"]
model_accuracy = accuracy_ks_coeffs_data["accuracy_data"]

# Average the coefficients across folds

In [14]:
average_coefs = {}
for session_name, folds in model_coeffs.items():
    # Stack the coefficients from all folds and compute the mean
    coefficients = np.array(list(folds.values()))  # Shape: (num_folds, num_features)
    average_coefs[session_name] = coefficients.mean(axis=0)  # Average across folds

# Plot the average accuracy across sessions 

In [ ]:
plot_accuracy_across_sessions(model_accuracy, mice_groups=mice_groups, plot_title="500ms before homing")

# Pull out the specific cell IDs of top cells so we can see what their receptive fields look like by going through the single cluster plots

In [ ]:
_14th_may_coefs = average_coefs["JAL8_14may"]
_7th_may_coefs = average_coefs["JAL8_flip3_7may"]

# Load the followng numpy arrau
_14_file = np.load(r"W:\branco\Laurence\JAL008\JAL008_shelter_barrier_flip_5_2024_05_14T10_18_03\processed_data\good_cluster_Ids.npy")
_7_file = np.load(r"W:\branco\Laurence\JAL008\JAL008_shelter_barrier_flip_3_2024_05_07T10_16_26\processed_data\good_cluster_Ids.npy")

# Acess the top positive 10 coefficients for the 14th may
_14th_may_indices = np.argsort(_14th_may_coefs)[::-1][:10]

print("The top 10 positive coefficients for the 14th may are", _14th_may_indices)
print("The top 10 positive coefficients for the 14th may are", _14th_may_coefs[_14th_may_indices])
print("The cell IDs for the 14th may are", sorted(_14_file[_14th_may_indices]))

# Acess the top positive 10 coefficients for the 7th may
_7th_may_indices = np.argsort(_7th_may_coefs)[::-1][:10]

print("The top 10 positive coefficients for the 7th may are", _7th_may_indices)
print("The top 10 positive coefficients for the 7th may are", _7th_may_coefs[_7th_may_indices])
print("The cell IDs for the 7th may are", sorted(_7_file[_7th_may_indices]))
